# Recognition, detection and segmentation · Module 11
Python 3 and NumPy only. Each part follows a lesson in this module: a CNN and transfer learning (41), attention and residual connections (55), boxes (42), masks (43), metrics (44) and multi-scale features (56). Tasks are small and synthetic so that the mechanics stay visible. Practical Labs 04-08 repeat these ideas on real images.


In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(211)
def softmax(z):
    shifted = z - z.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)
def cross_entropy(z, y):
    shifted = z - z.max(axis=1, keepdims=True)
    return np.mean(np.log(np.exp(shifted).sum(axis=1)) - shifted[np.arange(len(y)), y])
def make_bars(n, noise=.1):
    y = rng.integers(0, 2, n); images = np.zeros((n, 8, 8))
    for i, label in enumerate(y):
        p = int(rng.integers(1, 6))
        if label == 0: images[i, p:p+2, :] = 1
        else:          images[i, :, p:p+2] = 1
    return images + rng.normal(0, noise, images.shape), y
def patches(X): return np.lib.stride_tricks.sliding_window_view(X, (3, 3), axis=(1, 2)).reshape(len(X), 6, 6, 9)
X, y = make_bars(240)
Ptr, Pval, Ptest = patches(X[:160]), patches(X[160:200]), patches(X[200:])
ytr, yval, ytest = y[:160], y[160:200], y[200:]
parameters = {'K': rng.normal(0, .25, (9, 4)), 'bc': np.zeros(4), 'W': rng.normal(0, .05, (144, 2)), 'b': np.zeros(2)}
def cnn(P, p):
    Z = np.einsum('nijk,kf->nijf', P, p['K']) + p['bc']
    A = np.maximum(0, Z).reshape(len(P), -1)
    return Z, A, A @ p['W'] + p['b']
def loss_grad(P, y, p):
    Z, A, logits = cnn(P, p)
    G = (softmax(logits) - np.eye(2)[y]) / len(y)
    Gconv = (G @ p['W'].T).reshape(Z.shape) * (Z > 0)
    return cross_entropy(logits, y), {'W': A.T @ G, 'b': G.sum(axis=0), 'K': np.einsum('nijk,nijf->kf', P, Gconv), 'bc': Gconv.sum(axis=(0, 1, 2))}
print('trainable parameters:', sum(v.size for v in parameters.values()))
assert sum(v.size for v in parameters.values()) == 330


trainable parameters: 330


## Part 1: A shared-kernel derivative, then training
The kernel gradient accumulates contributions over the batch and over every placement. Check selected coordinates before training, as you did for the MLP.


In [2]:
analytic = loss_grad(Ptr[:4], ytr[:4], parameters)[1]
for name, index in [('K', (1, 2)), ('bc', (1,)), ('W', (17, 0)), ('b', (0,))]:
    value = parameters[name][index]; h = 1e-5
    parameters[name][index] = value + h; plus = loss_grad(Ptr[:4], ytr[:4], parameters)[0]
    parameters[name][index] = value - h; minus = loss_grad(Ptr[:4], ytr[:4], parameters)[0]
    parameters[name][index] = value
    error = abs((plus - minus) / (2 * h) - analytic[name][index])
    print('CNN gradient check:', name, index, error)
    assert error < 1e-6
initial = loss_grad(Ptr, ytr, parameters)[0]
for step in range(300):
    _, grads = loss_grad(Ptr, ytr, parameters)
    for name in parameters: parameters[name] -= .1 * grads[name]
final = loss_grad(Ptr, ytr, parameters)[0]
assert final < initial
print('training loss:', initial, 'to', final, ' validation accuracy:', np.mean(cnn(Pval, parameters)[2].argmax(axis=1) == yval))
print('fixed procedure, test accuracy:', np.mean(cnn(Ptest, parameters)[2].argmax(axis=1) == ytest), ' N:', len(ytest))


CNN gradient check: K (1, 2) 2.3891340295012498e-12
CNN gradient check: bc (1,) 4.2920944576252396e-12
CNN gradient check: W (17, 0) 3.091448971814792e-12
CNN gradient check: b (0,) 2.357614103942751e-12
training loss: 0.7021638510272182 to 0.0031808865039235938  validation accuracy: 1.0
fixed procedure, test accuracy: 1.0  N: 40


## Part 2: Transfer learning, and when it helps
Train a small pooled CNN on plenty of clean bars. Then reuse its kernels, frozen, for a new task with only 6 labelled images, so that only a tiny head trains. Compare that with training everything from those same 6 images. Two new tasks test the idea from both sides: one shares the old features, and one needs something else. The numbers are averages over 12 random draws of the labelled images. Practical Lab 05 repeats this with ResNet18, a far deeper network with far more to reuse.


In [3]:
def windows(X): return np.lib.stride_tricks.sliding_window_view(X, (3, 3), axis=(1, 2)).reshape(len(X), 36, 9)
def pooled(P, K, c): return np.maximum(0, P @ K + c).mean(axis=1)
def train_pooled(P, y, generator, steps, lr=1., kernels=4):
    K = generator.normal(0, .3, (9, kernels)); c = np.zeros(kernels); W = generator.normal(0, .05, (kernels, 2)); b = np.zeros(2); Y = np.eye(2)[y]
    for _ in range(steps):
        Z = P @ K + c; A = np.maximum(0, Z); F = A.mean(axis=1)
        G = (softmax(F @ W + b) - Y) / len(P); Gc = (G @ W.T)[:, None, :] * (Z > 0) / 36
        K -= lr * np.einsum('npk,npf->kf', P, Gc); c -= lr * Gc.sum(axis=(0, 1)); W -= lr * F.T @ G; b -= lr * G.sum(axis=0)
    return K, c, W, b
def fit_head(K, c, Ps, ys, steps=500, lr=2.):
    """Only the head changes here: K and c are frozen."""
    F = pooled(Ps, K, c); mu, sd = F.mean(axis=0), F.std(axis=0) + 1e-8
    A = (F - mu) / sd; W = np.zeros((K.shape[1], 2)); b = np.zeros(2); Y = np.eye(2)[ys]
    for _ in range(steps):
        G = (softmax(A @ W + b) - Y) / len(A)
        W -= lr * (A.T @ G + .02 * W); b -= lr * G.sum(axis=0)
    return lambda P: ((pooled(P, K, c) - mu) / sd) @ W + b
def bars_task(generator, n, noise):
    images = np.zeros((n, 8, 8)); y = generator.integers(0, 2, n)
    for i, label in enumerate(y):
        p = int(generator.integers(1, 6))
        if label == 0: images[i, p:p+2, :] = 1
        else:          images[i, :, p:p+2] = 1
    return images + generator.normal(0, noise, images.shape), y
def cross_task(generator, n, noise):
    """A different question: one bar, or a cross made of two?"""
    images = np.zeros((n, 8, 8)); y = generator.integers(0, 2, n)
    for i, label in enumerate(y):
        a, b = generator.integers(1, 6, 2)
        if label == 1: images[i, a:a+2, :] = 1; images[i, :, b:b+2] = 1
        elif generator.random() < .5: images[i, a:a+2, :] = 1
        else: images[i, :, b:b+2] = 1
    return images + generator.normal(0, noise, images.shape), y
source = np.random.default_rng(1); Xsrc, ysrc = bars_task(source, 300, .3)
K, c, _, _ = train_pooled(windows(Xsrc), ysrc, source, 500)
frozen = K.copy()
compare = {}
for name, task, noise in [('shares the old features: bars in heavy noise', bars_task, .9), ('needs something else: bar or cross', cross_task, .3)]:
    reused, scratch = [], []
    for trial in range(12):
        generator = np.random.default_rng(900 + trial); X, y = task(generator, 206, noise)
        Ps, Pt, ys, yt = windows(X[:6]), windows(X[6:]), y[:6], y[6:]
        if len(set(ys)) < 2: continue
        reused.append(np.mean(fit_head(K, c, Ps, ys)(Pt).argmax(axis=1) == yt))
        k2, c2, w2, b2 = train_pooled(Ps, ys, generator, 400)
        scratch.append(np.mean((pooled(Pt, k2, c2) @ w2 + b2).argmax(axis=1) == yt))
    compare[name] = np.mean(reused), np.mean(scratch)
    print(f'{name:46s} frozen pretrained {compare[name][0]:.3f}   trained from scratch {compare[name][1]:.3f}')
assert np.array_equal(frozen, K)
print('head parameters updated:', 4 * 2 + 2, 'of', K.size + c.size + 4 * 2 + 2)
same, other = compare.values()
assert same[0] > same[1] and other[0] < other[1]


shares the old features: bars in heavy noise   frozen pretrained 0.750   trained from scratch 0.654
needs something else: bar or cross             frozen pretrained 0.741   trained from scratch 0.863
head parameters updated: 10 of 50


Read those two rows carefully. Reusing features helps when the new task shares them, and it can hurt when the task needs something else, because frozen kernels cannot adapt. "Pretrained" is not automatically better, so compare it on your own split.


## Part 3: One attention head, and a residual connection
Attention scores are dot products, softmaxed along each row, then used to average the values. Dividing by the square root of d keeps scores at a sensible scale. A shuffle check shows why positional information has to be added separately. Then compare gradients through 20 layers with and without residual connections.


In [4]:
def attention(Q, K, V):
    scores = Q @ K.T / np.sqrt(Q.shape[1])
    weights = softmax(scores)
    return weights @ V, weights
d = 4; tokens = rng.normal(0, 1, (6, 16))
WQ, WK, WV = rng.normal(0, 1, (3, 16, d)) / 4
Q, K, V = tokens @ WQ, tokens @ WK, tokens @ WV
output, weights = attention(Q, K, V)
print('tokens', tokens.shape, ' attention weights', weights.shape, ' output', output.shape)
assert np.allclose(weights.sum(axis=1), 1)
order = rng.permutation(6)
assert np.allclose(attention(Q[order], K[order], V[order])[0], output[order])
print('shuffling tokens only shuffles outputs: without position, attention sees a set.')
q = np.array([[1., 0., 2., 1.]]); k = np.array([[2., 0., 0., 0.], [0., 1., 1., 1.]]); v = np.array([[4.], [-2.]])
print('hand-checkable row: weights', attention(q, k, v)[1], ' output', attention(q, k, v)[0])
slopes = rng.uniform(-.6, .6, 20)
plain, residual = np.prod(slopes), np.prod(1 + slopes)
print('gradient through 20 plain layers:', plain, ' through 20 residual layers:', residual)
assert abs(residual) > 1e3 * abs(plain)


tokens (6, 16)  attention weights (6, 6)  output (6, 4)
shuffling tokens only shuffles outputs: without position, attention sees a set.
hand-checkable row: weights [[0.3775 0.6225]]  output [[0.2652]]
gradient through 20 plain layers: 1.3072205991201026e-18  through 20 residual layers: 0.18974560642137356


## Part 4: Boxes, IoU and non-maximum suppression
Predict one box with a linear model, and compare it with always guessing the average box: a baseline that shows whether the model has learned anything about position. Then remove duplicate detections with NMS. Boxes use continuous boundary coordinates, without a +1.


In [5]:
def box_iou(a, b):
    left = np.maximum(a[..., :2], b[..., :2]); right = np.minimum(a[..., 2:], b[..., 2:])
    intersection = np.prod(np.maximum(0, right - left), axis=-1)
    union = np.prod(a[..., 2:] - a[..., :2], axis=-1) + np.prod(b[..., 2:] - b[..., :2], axis=-1) - intersection
    return np.where(union > 0, intersection / np.maximum(union, 1e-12), 0.)
assert np.isclose(box_iou(np.array([0., 0., 2., 2.]), np.array([1., 1., 3., 3.])), 1 / 7)
def scene(n, size=16):
    images = np.zeros((n, size, size)); boxes = np.zeros((n, 4))
    for i in range(n):
        h, w = rng.integers(4, 9, 2); top, left = rng.integers(0, size - h + 1), rng.integers(0, size - w + 1)
        images[i, top:top+h, left:left+w] = 1; boxes[i] = [left, top, left + w, top + h]
    return (images + rng.normal(0, .1, images.shape)).reshape(n, -1), boxes
Xb, boxes = scene(500)
design = np.c_[Xb, np.ones(len(Xb))]
ridge = np.linalg.solve(design[:400].T @ design[:400] + np.eye(design.shape[1]), design[:400].T @ boxes[:400])
predicted = design[400:] @ ridge
average = np.tile(boxes[:400].mean(axis=0), (100, 1))
print('mean IoU, always the average box:', box_iou(average, boxes[400:]).mean())
print('mean IoU, linear box regression: ', box_iou(predicted, boxes[400:]).mean())
assert box_iou(predicted, boxes[400:]).mean() > box_iou(average, boxes[400:]).mean() + .2
def nms(boxes, scores, threshold=.5):
    order = list(np.argsort(scores)[::-1]); keep = []
    while order:
        best = order.pop(0); keep.append(int(best))
        order = [i for i in order if box_iou(boxes[best], boxes[i]) < threshold]
    return keep
detections = np.array([[0., 0., 4., 4.], [.3, .2, 4.2, 4.1], [8., 8., 12., 12.], [.5, 0., 4.5, 4.]])
print('NMS keeps boxes', nms(detections, np.array([.9, .8, .7, .6])))
assert nms(detections, np.array([.9, .8, .7, .6])) == [0, 2]


mean IoU, always the average box: 0.1616341115139997
mean IoU, linear box regression:  0.4438349964648205
NMS keeps boxes [0, 2]


## Part 5: Masks, precision and recall
One scalar logit per pixel models bright synthetic foreground, which is deliberately simpler than an encoder-decoder. Compare it with the all-background baseline: accuracy flatters that baseline, and IoU does not. The AP here is a noninterpolated toy AP at one matching threshold, not COCO AP.


In [6]:
def rectangles(n):
    masks = np.zeros((n, 16, 16), dtype=bool)
    for i in range(n):
        top, left = rng.integers(0, 12, 2); height, width = rng.integers(2, 5, 2)
        masks[i, top:top+height, left:left+width] = True
    return .8 * masks + rng.normal(0, .12, masks.shape), masks
observed, masks = rectangles(140)
train, test, ttr, ttest = observed[:100], observed[100:], masks[:100], masks[100:]
a, bias = 1., 0.
for _ in range(500):
    z = a * train + bias; p = 1 / (1 + np.exp(-z)); residual = p - ttr
    a -= .8 * np.mean(residual * train); bias -= .8 * np.mean(residual)
def mask_metrics(pred, true):
    tp = np.sum(pred & true); fp = np.sum(pred & ~true); fn = np.sum(~pred & true)
    return dict(accuracy=round(float(np.mean(pred == true)), 3), iou=round(float(tp / (tp + fp + fn)) if tp + fp + fn else 0., 3), dice=round(float(2 * tp / (2 * tp + fp + fn)) if tp + fp + fn else 0., 3))
print('foreground share of pixels:', ttest.mean().round(3))
print('all background:', mask_metrics(np.zeros_like(ttest), ttest))
print('learned pixels:', mask_metrics(a * test + bias >= 0, ttest))
assert mask_metrics(np.zeros_like(ttest), ttest)['accuracy'] > .85 and mask_metrics(np.zeros_like(ttest), ttest)['iou'] == 0
assert mask_metrics(a * test + bias >= 0, ttest)['iou'] > .5
tp = np.array([1, 0, 1]); precision = np.cumsum(tp) / np.arange(1, 4); ap = precision[tp == 1].sum() / 2
assert np.isclose(ap, 5 / 6)
print('ranked precision:', precision, ' toy AP:', ap)


foreground share of pixels: 0.037
all background: {'accuracy': 0.963, 'iou': 0.0, 'dice': 0.0}
learned pixels: {'accuracy': 0.995, 'iou': 0.868, 'dice': 0.93}
ranked precision: [1.     0.5    0.6667]  toy AP: 0.8333333333333333


## Part 6: Multi-scale features
A Laplacian pyramid stores what each coarser level loses, so it rebuilds the image exactly. That is the idea behind a U-Net skip connection: coarse context alone loses fine detail, and the skip brings it back. The FPN rule then assigns each box to the level whose stride suits its size.


In [7]:
def down(x): return (x[::2, ::2] + x[1::2, ::2] + x[::2, 1::2] + x[1::2, 1::2]) / 4
def up(x): return np.repeat(np.repeat(x, 2, axis=0), 2, axis=1)
image = np.zeros((32, 32)); image[10:22, 14:16] = 1.; image[4:8, 4:28] = .5
levels = [image]
for _ in range(3): levels.append(down(levels[-1]))
laplacian = [levels[i] - up(levels[i + 1]) for i in range(3)]
rebuilt = levels[-1]
for detail in reversed(laplacian): rebuilt = up(rebuilt) + detail
print('pixels at each level:', [level.size for level in levels])
print('reconstruction error with every level:', np.abs(rebuilt - image).max())
assert np.allclose(rebuilt, image)
coarse_only = up(up(up(levels[-1])))
print('error using coarse context alone (no skips):', np.abs(coarse_only - image).max())
def fpn_level(w, h): return int(np.clip(np.floor(4 + np.log2(np.sqrt(w * h) / 224)), 2, 5))
for side in [32, 96, 224, 448, 900]:
    print(f'box {side}x{side}: level P{fpn_level(side, side)}')
assert fpn_level(96, 96) == 2 and fpn_level(224, 224) == 4


pixels at each level: [1024, 256, 64, 16]
reconstruction error with every level: 0.0
error using coarse context alone (no skips): 0.8125
box 32x32: level P2
box 96x96: level P2
box 224x224: level P4
box 448x448: level P5
box 900x900: level P5


## Your experiment
Declare one comparison before you run it: change bar noise, kernel count, the number of labelled images in Part 2 or the rectangle contrast. Keep target transformations aligned with input transformations. Use fresh held-out data for a revised procedure, and report coordinates, counts and metric conventions. Then move to Practical Labs 04-08, where the same ideas meet real images.
